# ViT Attention Diversity: `ones` vs `spectral` Input Mode

This notebook shows why **uniform constant inputs** (`ones`) produce degenerate attention patterns in Vision Transformers, while **per-band spectral inputs** (`spectral`) -- approximating Sentinel-2 surface-reflectance statistics -- produce more realistic, content-driven attention.

This corresponds to the `--data-mode` flag added to [throughput-bench](https://github.com/calebrob6/throughput-bench).

**Runtime:** GPU recommended (T4 free tier is fine); CPU works too.

In [ ]:
!pip install -q timm

In [ ]:
import torch
import timm
import numpy as np
import matplotlib.pyplot as plt

torch.manual_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device : {device}')
print(f'PyTorch: {torch.__version__}')
print(f'timm   : {timm.__version__}')

## 1 · Synthetic input modes

In [ ]:
# Sentinel-2 L2A surface-reflectance statistics (0-1 scale)
# Band order: B01 B02 B03 B04 B05 B06 B07 B08 B8A B09 B10 B11 B12
S2_BAND_NAMES = ['B01','B02','B03','B04','B05','B06','B07','B08','B8A','B09','B10','B11','B12']
S2_MEANS = [0.080,0.083,0.102,0.085,0.138,0.199,0.228,0.237,0.244,0.248,0.010,0.147,0.082]
S2_STDS  = [0.038,0.043,0.051,0.053,0.064,0.085,0.096,0.103,0.106,0.102,0.012,0.091,0.053]

MODES  = ['ones', 'randn', 'spectral']
COLORS = {'ones': '#e74c3c', 'randn': '#3498db', 'spectral': '#2ecc71'}


def make_batch(mode: str, batch_size: int = 4, channels: int = 13, size: int = 224) -> torch.Tensor:
    if mode == 'spectral':
        mu  = torch.tensor(S2_MEANS[:channels], dtype=torch.float32).view(1, channels, 1, 1)
        sig = torch.tensor(S2_STDS[:channels],  dtype=torch.float32).view(1, channels, 1, 1)
        return (mu + sig * torch.randn(batch_size, channels, size, size)).to(device)
    elif mode == 'randn':
        return torch.randn(batch_size, channels, size, size).to(device)
    else:  # ones
        return torch.ones(batch_size, channels, size, size).to(device)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)

for ax, mode in zip(axes, MODES):
    img   = make_batch(mode, batch_size=1).squeeze(0).cpu()
    means = img.mean(dim=(1, 2)).numpy()
    stds  = img.std(dim=(1, 2)).numpy()
    ax.bar(range(13), means, yerr=stds, capsize=3,
           color=COLORS[mode], alpha=0.85, ecolor='gray')
    ax.set_title(f"mode='{mode}'", fontsize=12, fontweight='bold')
    ax.set_xticks(range(13))
    ax.set_xticklabels(S2_BAND_NAMES, rotation=45, fontsize=8)
    ax.axhline(0, color='black', linewidth=0.5)

axes[0].set_ylabel('Mean pixel value (+/- 1 std)', fontsize=10)
fig.suptitle('Per-band statistics of synthetic inputs -- single image per mode', fontsize=13)
plt.tight_layout()
plt.show()
print()
print('ones    : all bands identical, zero spatial variance')
print('randn   : zero mean, unit variance -- no physical meaning')
print('spectral: each band has its own mean & std, matching S2 surface reflectance')

## 2 · Patch token diversity

Before the model runs, we can measure how much information the patch tokens carry. If all patch tokens are identical, no amount of attention can distinguish them.

We compute the **variance of patch tokens** across the spatial dimension (averaged over channels). Zero variance means every patch looks the same to the model before positional embeddings are added.

In [ ]:
PATCH_SIZE = 16


def patch_token_variance(x: torch.Tensor) -> torch.Tensor:
    B, C, H, W = x.shape
    n = H // PATCH_SIZE
    patches = x.unfold(2, PATCH_SIZE, PATCH_SIZE).unfold(3, PATCH_SIZE, PATCH_SIZE)
    tokens  = patches.contiguous().view(B, C, n * n, -1).mean(-1).permute(0, 2, 1)
    return tokens.var(dim=1).mean(dim=-1)  # (B,)


print(f"{'Mode':<10} {'Patch-token variance':>22}")
print('-' * 34)
for mode in MODES:
    var  = patch_token_variance(make_batch(mode).cpu()).numpy()
    note = '  <- all tokens identical' if var.mean() < 1e-8 else ''
    print(f'{mode:<10} {var.mean():>20.6f}{note}')

## 3 · ViT attention analysis

We load a randomly-initialised ViT-Small (13 input channels for S2) and register a forward hook on the **last block's attention-dropout layer**. Its input is the post-softmax attention weight matrix of shape `(B, H, N, N)`.

In [ ]:
model = timm.create_model(
    'vit_small_patch16_224',
    pretrained=False,
    in_chans=13,
    num_classes=10,
    img_size=224,
).to(device).eval()

n_params  = sum(p.numel() for p in model.parameters()) / 1e6
n_patches = (224 // PATCH_SIZE) ** 2  # 196
n_heads   = model.blocks[-1].attn.num_heads
max_ent   = float(np.log(n_patches))

print(f'ViT-S/16 ({n_params:.1f}M params) | {n_patches} patch tokens + 1 CLS | {n_heads} heads/block')
print(f'Max entropy (uniform over {n_patches} patches): {max_ent:.3f} nats')

captured: dict = {}


def _hook(module, inp, _out):
    captured['attn'] = inp[0].detach().cpu()


_handle = model.blocks[-1].attn.attn_drop.register_forward_hook(_hook)
print('Hook registered on model.blocks[-1].attn.attn_drop')

In [ ]:
BATCH_SIZE = 4
attn_maps: dict[str, torch.Tensor] = {}

with torch.no_grad():
    for mode in MODES:
        _ = model(make_batch(mode, batch_size=BATCH_SIZE))
        attn_maps[mode] = captured['attn'].clone()  # (B, H, N, N)

print('Captured:', {m: tuple(v.shape) for m, v in attn_maps.items()})

### Cross-image attention similarity

With `ones` inputs every image is identical, so every image produces **the exact same attention map**. The cross-image cosine-similarity matrix should be all 1s. With `spectral`, different images produce different attention patterns.

In [ ]:
def cls_attn(attn: torch.Tensor) -> np.ndarray:
    return attn[:, :, 0, 1:].numpy()  # (B, H, N_patches)


def pairwise_cosine(mat: np.ndarray) -> np.ndarray:
    n = np.linalg.norm(mat, axis=-1, keepdims=True) + 1e-9
    m = mat / n
    return m @ m.T


fig, axes = plt.subplots(1, 3, figsize=(13, 4))

for ax, mode in zip(axes, MODES):
    ca        = cls_attn(attn_maps[mode])
    B, H, P   = ca.shape
    sim       = pairwise_cosine(ca.reshape(B, H * P))
    off       = sim[~np.eye(B, dtype=bool)].mean()
    im = ax.imshow(sim, vmin=0.9, vmax=1.0, cmap='RdYlGn')
    ax.set_title(f"mode='{mode}'\nmean off-diag sim: {off:.4f}", fontsize=11)
    ax.set_xlabel('Image index')
    ax.set_ylabel('Image index')
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

fig.suptitle(
    'Cross-image cosine similarity of CLS->patch attention (last block)\n'
    '1.0 = identical attention maps across images',
    fontsize=12,
)
plt.tight_layout()
plt.show()

print()
for mode in MODES:
    ca      = cls_attn(attn_maps[mode])
    B, H, P = ca.shape
    sim     = pairwise_cosine(ca.reshape(B, H * P))
    off     = sim[~np.eye(B, dtype=bool)].mean()
    print(f'  {mode:<10}: mean off-diagonal similarity = {off:.6f}')

### CLS->patch attention maps

One row per mode, one column per attention head. Brighter = higher attention weight. Title shows per-head entropy in nats.

In [ ]:
patch_grid = 224 // PATCH_SIZE  # 14

fig, axes = plt.subplots(len(MODES), n_heads, figsize=(n_heads * 2.0, len(MODES) * 2.3))

for row, mode in enumerate(MODES):
    ca   = cls_attn(attn_maps[mode])[0]  # first image: (H, P)
    ents = -(ca * np.log(ca + 1e-9)).sum(axis=-1)
    for h in range(n_heads):
        ax = axes[row, h]
        ax.imshow(ca[h].reshape(patch_grid, patch_grid), cmap='hot', vmin=0)
        ax.set_title(f'H{h}\n{ents[h]:.2f}n', fontsize=7)
        ax.axis('off')
    axes[row, 0].set_ylabel(f"'{mode}'", fontsize=10, rotation=0, labelpad=52, va='center')

fig.suptitle(
    f'CLS->patch attention -- ViT-S/16 last block (first image in batch)\n'
    f'rows: input mode  |  cols: head  |  title: entropy (nats, max={max_ent:.2f})',
    fontsize=11,
)
plt.tight_layout()
plt.show()

### Attention entropy distribution

Entropy quantifies how *spread* the attention is over patches. Maximum entropy (perfectly uniform) = log(196) ~= 5.28 nats. The key issue with `ones` is not entropy magnitude -- it is that **all images share the same pattern** regardless of content.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))

for mode in MODES:
    ca  = cls_attn(attn_maps[mode])
    ent = -(ca * np.log(ca + 1e-9)).sum(axis=-1).flatten()
    ax.hist(ent, bins=20, alpha=0.7, color=COLORS[mode], label=f"'{mode}'", density=True)

ax.axvline(max_ent, color='black', ls='--', lw=1.2, label=f'max (uniform) = {max_ent:.2f} nats')
ax.set_xlabel('CLS->patch attention entropy (nats)', fontsize=12)
ax.set_ylabel('Density', fontsize=12)
ax.set_title('Attention entropy -- ViT-S/16 last block, all images x heads', fontsize=13)
ax.legend()
plt.tight_layout()
plt.show()

print(f"\n{'Mode':<10} {'Mean entropy (nats)':>22} {'% of max':>10}")
print('-' * 46)
for mode in MODES:
    ca = cls_attn(attn_maps[mode])
    me = -(ca * np.log(ca + 1e-9)).sum(axis=-1).mean()
    print(f'{mode:<10} {me:>22.4f} {100 * me / max_ent:>9.1f}%')

## Summary

| Metric | `ones` | `randn` | `spectral` |
|---|---|---|---|
| Per-band variance | 0 -- all channels identical | ~1.0 (unit normal) | Varies by band (S2 stats) |
| Patch token diversity | None before pos-embed | High | Moderate, physically motivated |
| Cross-image attention sim. | ~1.0 -- every image gets the same map | < 1.0 | < 1.0 |
| Attention driven by | Positional biases only | Noise + position | Content + position |

**Why this matters for benchmarking geospatial ViTs:**

- `ones` makes all patch tokens identical before positional embeddings. Every image in the batch produces the **exact same attention map** -- the model is not exercising content-driven attention paths at all.
- `randn` adds token diversity but with statistics bearing no resemblance to satellite imagery (zero mean, unit variance, no inter-band correlation).
- `spectral` uses per-band means and stds derived from Sentinel-2 L2A surface reflectance. Different images get different attention patterns, and band magnitudes reflect real sensor output.

**Use `--data-mode spectral` when benchmarking geospatial ViTs:**

```bash
python benchmark.py --gpu-id 0 --data-mode spectral
```